In [7]:
import sys
import os
import torch.nn.functional as F
project_root = os.path.abspath(os.path.join(os.getcwd(),'..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
from src.processor.word_process import sudachi_tokenize
import pandas as pd
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F
from sklearn.metrics import accuracy_score

In [11]:
df = pd.read_csv("../data/raw/industry_master_33_refined.csv")
label_names = ["水産・農林業", "鉱業","建設業","食料品","繊維製品","パルプ・紙", "化学", "医薬品", "石油・石炭製品", "ゴム製品", "ガラス・土石製品", "鉄鋼", "非鉄金属", "金属製品", "機械", "電気機器", "輸送用機器", "精密機器", "その他製品", "電気・ガス業", "陸運業", "海運業", "空運業", "倉庫・運輸関連業", "情報・通信業", "卸売業", "小売業", "銀行業", "証券、商品先物取引業", "保険業", "その他金融業", "不動産業", "サービス業"]
label_description = df["業種説明"]

0     本業種は、農業、林業、および水産養殖業を含む漁業に関する事業を行う企業が分類される業種である...
1     本業種は、天然に固体・液体・ガスの状態で存在する鉱物を掘採し、選鉱や品位向上処理を行う事業で...
2     本業種は、土木施設や建築物の建設工事を発注者から直接請け負う、あるいは自ら手掛ける事業を行う...
3     本業種は、畜産食料品、水産食料品、農産加工品、調味料、精穀・製粉、パン・菓子、めん類、豆腐、...
4     本業種は、製糸、紡績、織物、ニット生地、染色整理、縫製など、糸や生地から繊維製品を製造する事...
5     本業種は、木材やその他の植物原料、古紙などを原料としてパルプおよび紙を製造し、さらに板紙、加...
6     本業種は、原油や天然ガス、鉱物などを原料として化学的処理を行い、樹脂、合成繊維原料、塗料、界...
7     本業種は、医薬品の原薬・原液から製剤に至るまでの製造・販売を行う事業である。具体的には、医療...
8     本業種は、原油を精製してガソリン、灯油、軽油、重油、潤滑油、グリースなどの石油製品を製造する...
9     本業種は、天然ゴムや合成ゴムを原料として、タイヤ、チューブ、工業用ゴム製品、ゴムベルト、ゴム...
10    本業種は、板ガラスやガラス製品、セメント、生コンクリート、陶磁器、耐火物、炭素・黒鉛製品、研...
11    本業種は、鉄鉱石やスクラップ（鉄くず）を原料として銑鉄や粗鋼を製造し、これを圧延・加工して鋼...
12    本業種は、銅、アルミニウム、亜鉛、鉛、ニッケルなど鉄以外の金属について、鉱石やスクラップから...
13    本業種は、鉄鋼や非鉄金属を素材として、建設用・建築用金属製品、金属線製品、刃物、手道具、一般...
14    本業種は、汎用的に他の機械へ組み込まれるはん用機械器具や、生産設備として用いられる生産用機械...
15    本業種は、電気エネルギーの発生・貯蔵・送電・変電・利用に関わる機械器具や、家庭用の民生用電気...
16    本業種は、自動車、二輪車、船舶、鉄道車両、航空機など、人や貨物を輸送するための機械器具を製造...
17    本業種は、高い精度の加工・組立技術を要する精密機械器具の製造・販売を行う事業である。

In [ ]:
# 1. NumPy配列をロード
business_np = np.load('../models/true_neo_company_embeddings.npy')
category_np = np.load('../models/true_neo_category_embeddings.npy')
# 2. PyTorch Tensorに変換
# torch.from_numpy() を使うのが最も効率的です
business_embeddings = torch.from_numpy(business_np)
category_embeddings = torch.from_numpy(category_np)
# 3. 必要に応じて型を合わせる
# モデルの出力（label_embsなど）が float32 の場合、型を合わせないと計算エラーになります
business_embeddings = business_embeddings.to(torch.float32)
category_embeddings = category_embeddings.to(torch.float32)
# 4. もしGPUを使いたいなら
# test_embeddings = test_embeddings.to('cuda')

In [13]:
final_33d_vector = torch.matmul(business_embeddings, category_embeddings.T)

In [14]:
final_33d_vector.shape

torch.Size([3637, 33])

In [15]:
# 1. データの読み込み
data = pd.read_csv("../data/raw/edinet_analysis_data_listed_only.csv")

# 2. TensorをNumPy配列に変換（GPUにある場合は.cpu()が必要）
vector_numpy = final_33d_vector.detach().cpu().numpy()

# 3. 33次元の各要素を「v_0, v_1, ... v_32」という列名でDataFrame化
cols = [f"v_{i}" for i in range(vector_numpy.shape[1])]
df_vector = pd.DataFrame(vector_numpy, columns=cols, index=data.index)

# 4. 元のデータと横方向に結合（マージ）
data_with_vector = pd.concat([data, df_vector], axis=1)

# 確認
print(data_with_vector.shape)
data_with_vector.head()

(3637, 42)


,company_name,証券コード,33業種コード,33業種区分,17業種コード,17業種区分,business_description,business_risks,filename,v_0,...,v_23,v_24,v_25,v_26,v_27,v_28,v_29,v_30,v_31,v_32
0,AGC株式会社,5201,3400,ガラス・土石製品,3,建設・資材,3 事業の内容 当社及び当社の子会社(以下、「当社グループ」という。)並びに当社の関連会社は...,3 事業等のリスク (1)リスクマネジメント体制 当社グループでは、「第4 提出会社の状況 ...,S100XSTU.zip,1.243053,...,-11.394192,12.404107,1.854251,-5.110681,-3.555357,0.573859,-4.443158,-4.586052,4.724018,-1.160245
1,AGS株式会社,3648,5250,情報・通信業,10,情報通信・サービスその他,3 事業の内容 当社グループ(当社及び当社の関係会社)は、当社と連結子会社3社とで構成されて...,3 事業等のリスク 文中の将来に関する事項は、当連結会計年度末現在において当社グループが判断...,S100W007.zip,24.916206,...,0.326024,83.876991,15.689158,22.079231,4.364710,9.540520,13.735349,8.277737,18.087534,28.978081
2,AHCグループ株式会社,7083,9050,サービス業,10,情報通信・サービスその他,3 事業の内容 当社グループは、当社、連結子会社(SLカンパニー株式会社、テラスワールド株式...,3 事業等のリスク 有価証券報告書に記載した事業の状況、経理の状況等に関する事項のうち、投資...,S100XMZ1.zip,27.887882,...,17.534468,16.013958,18.377945,22.710514,9.927882,4.650744,13.399891,8.244039,14.610176,46.568527
3,AI CROSS株式会社,4476,5250,情報・通信業,10,情報通信・サービスその他,"3 事業の内容 当社グループは、「Smart Work, Smart Life~人生のいい時...",3 事業等のリスク 本書に記載した事業の状況、経理の状況等に関する事項のうち、投資家の判断に...,S100XUJV.zip,24.258528,...,3.380911,82.237556,5.187076,11.194658,2.863756,7.913954,13.467419,6.765543,16.952774,40.013073
4,AI inside 株式会社,4488,5250,情報・通信業,10,情報通信・サービスその他,3 事業の内容 当社は、「AIで、人類の進化と人々の幸福に貢献する」というパーパスのもと、「...,3 事業等のリスク 当社は、事業展開上のリスクになる可能性があると考えられる主な要因として、...,S100W5FA.zip,24.018238,...,-1.091570,75.512604,7.946282,11.024022,1.952577,8.226843,9.016088,5.549695,14.556294,31.208803


In [16]:
# 1. 33次元のベクトル列（v_0 〜 v_32）だけを抽出してTensorに変換
# ※ index_col等に合わせて適切な企業名列（例: '企業名' や 'company_name' など）を指定してください
target_company = "KDDI株式会社"  # 確認したい企業名

if target_company in data_with_vector['company_name'].values:
    # 対象企業の行を抽出
    row_data = data_with_vector[data_with_vector['company_name'] == target_company].iloc[0]
    
    # 結合した v_0 〜 v_32 の値を抜き出して PyTorch の Tensor に戻す
    v_cols = [f"v_{i}" for i in range(33)]
    company_vector = torch.tensor(row_data[v_cols].values.astype(float))
    
    # 2. 業種名マスターのリスト（label_names）を用意
    # ※ お手元の industry_master_33 などの「業種名」列からリストを作成します
    # df_industry = pd.read_csv("../data/raw/industry_master_33.csv")
    # label_names = df_industry["業種名"].tolist()
    # もしすでに定義済みならそのままでOKです
    
    # 3. torch.topk で上位3つを抽出（セル45のロジック）
    values, indices = torch.topk(company_vector, k=5)
    
    print(f"【{target_company}】の分析結果")
    print("-" * 30)
    print(f"実際の業種: {row_data['33業種区分']}")
    print(f"第1位: {label_names[indices[0].item()]} (スコア: {values[0].item():.4f})")
    print(f"第2位: {label_names[indices[1].item()]} (スコア: {values[1].item():.4f})")
    print(f"第3位: {label_names[indices[2].item()]} (スコア: {values[2].item():.4f})")
    print(f"第4位: {label_names[indices[3].item()]} (スコア: {values[3].item():.4f})")
    print(f"第5位: {label_names[indices[4].item()]} (スコア: {values[4].item():.4f})")

else:
    print(f"指定された企業『{target_company}』が見つかりませんでした。")

【KDDI株式会社】の分析結果
------------------------------
実際の業種: 情報・通信業
第1位: 情報・通信業 (スコア: 55.9703)
第2位: 電気・ガス業 (スコア: 30.0920)
第3位: サービス業 (スコア: 23.6950)
第4位: 不動産業 (スコア: 22.4109)
第5位: 電気機器 (スコア: 19.9634)


In [17]:
# 1. 各企業の正解業種のインデックスを取得するマップを作成
name_to_idx = {name: i for i, name in enumerate(label_names)}
data_with_vector['true_idx'] = data_with_vector['33業種区分'].map(name_to_idx)

# 2. 予測確率の行列（3637, 33）を取得
v_cols = [f"v_{i}" for i in range(33)]
pred_matrix = data_with_vector[v_cols].values

results = []
for idx, row in data_with_vector.iterrows():
    true_idx = row['true_idx']
    if pd.isna(true_idx): continue
    
    # 正解業種に割り当てられた予測確率
    true_prob = pred_matrix[idx, int(true_idx)]
    
    # 修正：正解業種が何番目に高かったか（順位：1〜33）を計算
    # 自分の確率より大きい要素の個数 + 1 が順位になります
    rank = np.sum(pred_matrix[idx] > true_prob) + 1
    
    results.append({
        'company_name': row['company_name'],
        'actual_industry': row['33業種区分'],
        'true_industry_prob': true_prob,
        'true_industry_rank': rank
    })

df_eval = pd.DataFrame(results)

# 例：正解の順位が悪かった（モデルが大きく外した）企業トップ10を見る
df_eval.sort_values('true_industry_rank', ascending=False).head(10)

,company_name,actual_industry,true_industry_prob,true_industry_rank
266,アジア航測株式会社,空運業,1.050453,30
2819,株式会社フロンティア,不動産業,-0.785346,24
3235,株式会社広済堂ホールディングス,その他製品,3.139270,22
1344,日建工学株式会社,サービス業,-1.280141,22
1135,出光興産株式会社,石油・石炭製品,15.931993,21
3540,細谷火工株式会社,化学,9.665144,21
1971,株式会社アイフィスジャパン,その他製品,9.208618,20
2414,株式会社ジェイホールディングス,卸売業,3.144893,19
2246,株式会社キムラタン,繊維製品,20.956589,19
2059,株式会社イチネンホールディングス,サービス業,15.756176,18


In [18]:
# 予測第1位のインデックスを取得
pred_1st_idx = np.argmax(pred_matrix, axis=1)
true_indices = data_with_vector['true_idx'].astype(int).values

# Top-1 精度（どんぴしゃ正解率）
top1_acc = accuracy_score(true_indices, pred_1st_idx)
print(f"Top-1 Accuracy (1位的中率): {top1_acc:.4f}")

# Top-3 / Top-5 精度
top3_count = 0
top5_count = 0
for i in range(len(true_indices)):
    top_indices = np.argsort(-pred_matrix[i])
    if true_indices[i] in top_indices[:3]:
        top3_count += 1
    if true_indices[i] in top_indices[:5]:
        top5_count += 1

print(f"Top-3 Accuracy (3位以内): {top3_count / len(true_indices):.4f}")
print(f"Top-5 Accuracy (5位以内): {top5_count / len(true_indices):.4f}")

Top-1 Accuracy (1位的中率): 0.7520
Top-3 Accuracy (3位以内): 0.9076
Top-5 Accuracy (5位以内): 0.9535


In [23]:
model_path = "../models/e5_edinet_finetuned"

print("モデルを読み込み中...")
loaded_tokenizer = AutoTokenizer.from_pretrained(model_path)
loaded_model = AutoModelForSequenceClassification.from_pretrained(model_path)

# 予測したい新しい企業の「事業の内容」テキスト
# （例として適当な文を入れています。ここを自由に書き換えてください）
new_business_description = """
当社グループは、主に自動車用部品の製造および販売を行っており、
主要製品であるトランスミッションやブレーキシステムの開発において、
次世代の電気自動車（EV）向け軽量化技術の構築を進めております。
"""
# 【超重要】E5のルール：必ず先頭に "query: " をつける
input_text = f"query: {new_business_description}"

# テキストをトークン（数値）に変換
inputs = loaded_tokenizer(
    input_text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=512
)
# 予測（推論）の実行
loaded_model.eval()
with torch.no_grad():
    outputs = loaded_model(**inputs)
    # 33クラスの確率（logits）が最も高いインデックスを取得
    pred_id = torch.argmax(outputs.logits, dim=-1).item()

# モデルの設定に自動保存されている「id2label」を使って数値IDを業種名に変換
predicted_industry = loaded_model.config.id2label[pred_id]

print("\n" + "="*30)
print(f"◆ 予測された業種: {predicted_industry}")
print("="*30)
    
    

モデルを読み込み中...


Loading weights: 100%|████████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 594.53it/s]



◆ 予測された業種: 輸送用機器
